# KIOS VEPL Power Line Detection Training

This notebook trains a binary power line segmentation model on the preprocessed KIOS VEPL dataset.

## Model Configuration:
- **Architecture**: DeepLabV3 with ResNet34 encoder
- **Loss Function**: Combined Tversky + Weighted BCE (optimized for class imbalance)
- **Task**: Binary power line segmentation
- **Device**: MPS-optimized for Apple Silicon

## Training Strategy:
- Adjusted for small dataset (~300-500 tiles)
- Aggressive data augmentation
- Smaller batch size (8)
- More epochs (15-20)
- Early stopping to prevent overfitting

## 1. Import Required Libraries

In [1]:
import os
import cv2
import torch
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import random

print("All libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"Segmentation Models PyTorch version: {smp.__version__}")

All libraries imported successfully
PyTorch version: 2.7.1
Segmentation Models PyTorch version: 0.5.0


## 2. Configuration and Hyperparameters

In [2]:
# =============================================================================
# DEVICE CONFIGURATION
# =============================================================================
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# MPS-specific settings
if DEVICE == "mps":
    torch.backends.mps.allow_tf32 = True
    print("MPS optimizations enabled")

# =============================================================================
# DATA PATHS
# =============================================================================
PROCESSED_IMG_DIR = '../data/processed_data/images'
PROCESSED_MSK_DIR = '../data/processed_data/masks'
MODEL_SAVE_DIR = '../models'
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# =============================================================================
# TRAINING HYPERPARAMETERS - OPTIMIZED FOR SMALL DATASET
# =============================================================================
IMAGE_HEIGHT = 256
IMAGE_WIDTH = 256
RANDOM_SEED = 42

# Model architecture
ENCODER = "resnet34"
ARCHITECTURE = "DeepLabV3"

# Training configuration - adjusted for small dataset
BATCH_SIZE = 8  # Smaller batch for limited data
NUM_EPOCHS = 20  # More epochs for smaller dataset
LEARNING_RATE = 0.0001
WEIGHT_DECAY = 0.00001

# Tversky loss parameters (same as original)
TVERSKY_ALPHA = 0.3  # Weight for false positives
TVERSKY_BETA = 0.7   # Weight for false negatives (prioritizes recall)

# BCE positive class weight (handle class imbalance)
BCE_POS_WEIGHT = 15.0

print(f"\nModel Configuration:")
print(f"{ARCHITECTURE}+{ENCODER}")
print(f"Batch Size: {BATCH_SIZE} (small for limited data)")
print(f"Max Epochs: {NUM_EPOCHS} (increased for small dataset)")
print(f"Initial LR: {LEARNING_RATE}, Weight Decay: {WEIGHT_DECAY}")
print(f"Tversky Loss: α={TVERSKY_ALPHA}, β={TVERSKY_BETA}")
print(f"BCE Positive Weight: {BCE_POS_WEIGHT}x")

Using device: mps
MPS optimizations enabled

Model Configuration:
DeepLabV3+resnet34
Batch Size: 8 (small for limited data)
Max Epochs: 20 (increased for small dataset)
Initial LR: 0.0001, Weight Decay: 1e-05
Tversky Loss: α=0.3, β=0.7
BCE Positive Weight: 15.0x


## 3. Custom Loss Functions

In [3]:
class TverskyLoss(nn.Module):
    """
    Tversky Loss with focal component for extreme class imbalance.
    
    Tversky index generalizes Dice coefficient allowing different weightings
    of false positives and false negatives.
    
    Args:
        alpha: Weight for false positives
        beta: Weight for false negatives  
        smooth: Smoothing factor to avoid division by zero
    """
    def __init__(self, alpha=0.5, beta=0.5, smooth=1e-6):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Apply sigmoid to convert logits to probabilities
        inputs = torch.sigmoid(inputs)
        
        # Flatten tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        
        # Calculate True Positives, False Positives & False Negatives
        TP = (inputs * targets).sum()    
        FP = ((1-targets) * inputs).sum()
        FN = (targets * (1-inputs)).sum()
       
        # Calculate Tversky index
        denominator = TP + self.alpha*FP + self.beta*FN + self.smooth
        Tversky = (TP + self.smooth) / denominator
        
        # Add focal component to focus on hard examples
        focal_weight = torch.pow(1 - Tversky, 2)
        
        return focal_weight * (1 - Tversky)


class CombinedLoss(nn.Module):
    """
    Combined loss for extreme class imbalance in power line detection.
    Combines Focal Tversky loss with weighted BCE for better convergence.
    """
    def __init__(self, tversky_weight=0.7, bce_weight=0.3, pos_weight=10.0):
        super(CombinedLoss, self).__init__()
        self.tversky_weight = tversky_weight
        self.bce_weight = bce_weight
        self.tversky_loss = TverskyLoss(alpha=TVERSKY_ALPHA, beta=TVERSKY_BETA)
        self.bce_loss = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight))
    
    def forward(self, inputs, targets):
        tversky = self.tversky_loss(inputs, targets)
        bce = self.bce_loss(inputs, targets)
        return self.tversky_weight * tversky + self.bce_weight * bce

print("Loss functions defined successfully")
print(f"Combined loss: 70% Focal Tversky + 30% Weighted BCE")
print(f"Optimized for extreme class imbalance in power line detection")

Loss functions defined successfully
Combined loss: 70% Focal Tversky + 30% Weighted BCE
Optimized for extreme class imbalance in power line detection


## 4. Custom Dataset Class

In [4]:
class KIOSPowerLineDataset(Dataset):
    """
    Custom Dataset class for KIOS power line segmentation.
    Loads preprocessed RGB images and binary grayscale masks.
    """
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        
        # Load RGB image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Load binary mask (already preprocessed as grayscale)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        # Normalize mask to [0, 1]
        binary_mask = (mask > 0).astype(np.float32)
        binary_mask = np.expand_dims(binary_mask, axis=-1)

        # Apply transformations
        if self.transform:
            augmented = self.transform(image=image, mask=binary_mask)
            image = augmented['image']
            binary_mask = augmented['mask']
            binary_mask = binary_mask.permute(2, 0, 1)

        return image, binary_mask

print("KIOSPowerLineDataset class defined successfully")

KIOSPowerLineDataset class defined successfully


## 5. Training and Evaluation Functions

In [5]:
def train_fn(loader, model, optimizer, loss_fn, scaler=None):
    """
    Training function for one epoch.
    """
    model.train()
    loop = tqdm(loader, desc="Training")
    total_loss = 0.0
    num_batches = len(loader)
    
    use_amp = (DEVICE == "cuda")
    
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.to(device=DEVICE, non_blocking=True)
        targets = targets.float().to(device=DEVICE, non_blocking=True)
        
        # Skip batches that are too small for BatchNorm
        if data.size(0) < 2:
            continue

        # Forward pass
        if use_amp and scaler is not None:
            with torch.autocast(device_type="cuda", enabled=True):
                predictions = model(data)
                loss = loss_fn(predictions, targets)
        else:
            predictions = model(data)
            loss = loss_fn(predictions, targets)
        
        total_loss += loss.item()

        # Backward pass
        optimizer.zero_grad()
        
        if use_amp and scaler is not None:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        
        # MPS memory management
        if DEVICE == "mps" and batch_idx % 50 == 0:
            torch.mps.empty_cache()

        loop.set_postfix({
            'loss': f'{loss.item():.4f}',
            'avg_loss': f'{total_loss/(batch_idx+1):.4f}'
        })
        
    return total_loss / num_batches


def check_accuracy(loader, model, device):
    """
    Evaluation function with IoU, Dice, and pixel accuracy metrics.
    """
    model.eval()
    num_correct = 0
    num_pixels = 0
    dice_score = 0
    iou_score = 0
    valid_batches = 0

    with torch.no_grad():
        for batch_idx, (x, y) in enumerate(loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            
            if x.size(0) < 1:
                continue
            
            preds = torch.sigmoid(model(x))
            preds = (preds > 0.5).float()
            
            num_correct += (preds == y).sum()
            num_pixels += torch.numel(preds)
            
            tp, fp, fn, tn = smp.metrics.get_stats(preds.long(), y.long(), mode='binary')
            batch_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="micro")
            batch_dice = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")
            
            # Only add valid (non-NaN) scores
            if not torch.isnan(batch_iou):
                iou_score += batch_iou
                valid_batches += 1
            if not torch.isnan(batch_dice):
                dice_score += batch_dice
            
            # MPS memory management
            if DEVICE == "mps" and batch_idx % 50 == 0:
                torch.mps.empty_cache()

    # Handle case where all IoU values are NaN
    if valid_batches > 0:
        iou = iou_score / valid_batches
    else:
        iou = torch.tensor(0.0)
        print("⚠️  Warning: All IoU calculations returned NaN")
    
    dice = dice_score / len(loader)
    acc = num_correct / num_pixels * 100
    
    print(f"Validation: IoU={iou:.4f} | Dice={dice:.4f} | Accuracy={acc:.2f}%")

    model.train()
    return iou.item()

print("Training and evaluation functions defined successfully")

Training and evaluation functions defined successfully


## 6. Data Loading and Preparation

In [6]:
# Load all processed tile paths
print("Loading preprocessed KIOS VEPL data...")

all_image_paths = sorted([os.path.join(PROCESSED_IMG_DIR, f) for f in os.listdir(PROCESSED_IMG_DIR) if f.endswith('.png')])
all_mask_paths = sorted([os.path.join(PROCESSED_MSK_DIR, f) for f in os.listdir(PROCESSED_MSK_DIR) if f.endswith('.png')])

print(f"Found {len(all_image_paths)} image tiles")
print(f"Found {len(all_mask_paths)} mask tiles")

assert len(all_image_paths) == len(all_mask_paths), f"Mismatch: {len(all_image_paths)} images vs {len(all_mask_paths)} masks"
print("✅ Image and mask counts match")

# Dataset size warning
if len(all_image_paths) < 500:
    print(f"\n⚠️  WARNING: Small dataset ({len(all_image_paths)} tiles)")
    print("   Recommendations:")
    print("   - Using aggressive data augmentation")
    print("   - Monitoring for overfitting carefully")
    print("   - Consider labeling more images for better performance")

# Split data (60% train, 20% validation, 20% test)
train_val_imgs, test_imgs, train_val_msks, test_msks = train_test_split(
    all_image_paths, all_mask_paths, 
    test_size=0.2, 
    random_state=RANDOM_SEED
)

train_imgs, val_imgs, train_msks, val_msks = train_test_split(
    train_val_imgs, train_val_msks,
    test_size=0.25,  # 0.25 of 0.8 = 0.2 of total
    random_state=RANDOM_SEED
)

print(f"\nData split:")
print(f"Training: {len(train_imgs)} tiles ({len(train_imgs)/len(all_image_paths)*100:.1f}%)")
print(f"Validation: {len(val_imgs)} tiles ({len(val_imgs)/len(all_image_paths)*100:.1f}%)")
print(f"Test: {len(test_imgs)} tiles ({len(test_imgs)/len(all_image_paths)*100:.1f}%)")

Loading preprocessed KIOS VEPL data...
Found 2139 image tiles
Found 2139 mask tiles
✅ Image and mask counts match

Data split:
Training: 1283 tiles (60.0%)
Validation: 428 tiles (20.0%)
Test: 428 tiles (20.0%)


## 7. Data Augmentation Strategy

In [7]:
print("Defining AGGRESSIVE data augmentation for small dataset...")

# Aggressive augmentation for training (important for small datasets)
train_transform = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    
    # Geometric augmentations (critical for power lines)
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),  # Increased from 0.3
    A.RandomRotate90(p=0.5),
    A.Rotate(limit=20, p=0.5),  # Increased from 15
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    
    # Spectral augmentations
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),  # Increased
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),  # Increased
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),  # Add noise
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),  # Add blur
    
    # Normalization (ImageNet stats for pretrained ResNet34)
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
    ToTensorV2(),
])

# Validation transform (no augmentation)
val_transform = A.Compose([
    A.Resize(height=IMAGE_HEIGHT, width=IMAGE_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], max_pixel_value=255.0),
    ToTensorV2(),
])

print("AGGRESSIVE augmentation strategy for small dataset:")
print("- Geometric: HFlip, VFlip, Rotate90, Rotate, ShiftScaleRotate")
print("- Spectral: ColorJitter, BrightnessContrast, GaussNoise, GaussianBlur")
print("- Normalization: ImageNet stats for pretrained encoder")
print("\nThis helps prevent overfitting on small datasets")

Defining AGGRESSIVE data augmentation for small dataset...
AGGRESSIVE augmentation strategy for small dataset:
- Geometric: HFlip, VFlip, Rotate90, Rotate, ShiftScaleRotate
- Spectral: ColorJitter, BrightnessContrast, GaussNoise, GaussianBlur
- Normalization: ImageNet stats for pretrained encoder

This helps prevent overfitting on small datasets


/Users/user/Desktop/university/thesis/veg_enc_code/veg_enc_env/lib/python3.10/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/var/folders/ps/w_1s3b6515181dg8hd57mkd80000gn/T/ipykernel_62638/1622271878.py:17: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),  # Add noise


## 8. Initialize Model and Training Components

In [8]:
print("=" * 70)
print("KIOS VEPL POWER LINE SEGMENTATION MODEL")
print("=" * 70)

# Initialize model
print(f"\nInitializing {ARCHITECTURE} with {ENCODER} encoder...")
model = smp.DeepLabV3(
    encoder_name=ENCODER,
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,} total, {trainable_params:,} trainable")

# Initialize loss function
loss_fn = CombinedLoss(tversky_weight=0.7, bce_weight=0.3, pos_weight=BCE_POS_WEIGHT)
print(f"\nLoss: Combined (70% Focal Tversky + 30% Weighted BCE)")
print(f"BCE positive weight: {BCE_POS_WEIGHT}x (for class imbalance)")

# Initialize optimizer
if DEVICE == "mps":
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    print(f"\nOptimizer: AdamW (better for MPS)")
else:
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    print(f"\nOptimizer: Adam")

print(f"Learning Rate: {LEARNING_RATE}, Weight Decay: {WEIGHT_DECAY}")

# Learning Rate Scheduler with early stopping
from torch.optim.lr_scheduler import ReduceLROnPlateau
scheduler = ReduceLROnPlateau(
    optimizer, 
    mode='max',
    factor=0.5,
    patience=5,  # Increased patience for small dataset
    min_lr=1e-6,
)
print(f"\nLR Scheduler: ReduceLROnPlateau (patience=5, factor=0.5)")

# Initialize gradient scaler (CUDA only)
scaler = None
if DEVICE == "cuda":
    scaler = torch.cuda.amp.GradScaler()
    print("CUDA AMP enabled")
elif DEVICE == "mps":
    print("MPS: Native precision (no AMP)")

# Create data loaders
print(f"\nCreating data loaders (batch size: {BATCH_SIZE})...")

num_workers = 0 if DEVICE == "mps" else 4
pin_memory = False if DEVICE == "mps" else True

train_loader = DataLoader(
    KIOSPowerLineDataset(train_imgs, train_msks, transform=train_transform),
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=num_workers, 
    drop_last=True,
    pin_memory=pin_memory
)
val_loader = DataLoader(
    KIOSPowerLineDataset(val_imgs, val_msks, transform=val_transform),
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=num_workers, 
    drop_last=False,
    pin_memory=pin_memory
)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

# MPS memory management
if DEVICE == "mps":
    torch.mps.empty_cache()
    print("\nMPS cache cleared")

print(f"\n✅ Ready to train for up to {NUM_EPOCHS} epochs")
print("=" * 70)

KIOS VEPL POWER LINE SEGMENTATION MODEL

Initializing DeepLabV3 with resnet34 encoder...
Model parameters: 26,007,105 total, 26,007,105 trainable

Loss: Combined (70% Focal Tversky + 30% Weighted BCE)
BCE positive weight: 15.0x (for class imbalance)

Optimizer: AdamW (better for MPS)
Learning Rate: 0.0001, Weight Decay: 1e-05

LR Scheduler: ReduceLROnPlateau (patience=5, factor=0.5)
MPS: Native precision (no AMP)

Creating data loaders (batch size: 8)...
Training batches: 160
Validation batches: 54

MPS cache cleared

✅ Ready to train for up to 20 epochs


## 9. Training Loop with Early Stopping

In [9]:
# Training with early stopping
best_iou = -1
patience_counter = 0
early_stop_patience = 7  # Stop if no improvement for 7 epochs

training_history = {
    'train_loss': [],
    'val_iou': [],
    'learning_rates': []
}

print("\nStarting training with early stopping...")
print(f"Early stopping patience: {early_stop_patience} epochs\n")

for epoch in range(NUM_EPOCHS):
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | LR: {current_lr:.2e}")
    
    # MPS memory management
    if DEVICE == "mps":
        torch.mps.empty_cache()
    
    # Training phase
    train_loss = train_fn(train_loader, model, optimizer, loss_fn, scaler)
    
    # Validation phase
    current_iou = check_accuracy(val_loader, model, device=DEVICE)
    
    # Update learning rate scheduler
    scheduler.step(current_iou)
    
    # Save training history
    training_history['train_loss'].append(train_loss)
    training_history['val_iou'].append(current_iou)
    training_history['learning_rates'].append(current_lr)
    
    # Check if learning rate was reduced
    new_lr = optimizer.param_groups[0]['lr']
    if new_lr < current_lr:
        print(f"📉 Learning rate reduced: {current_lr:.2e} → {new_lr:.2e}")
    
    # Save best model and check for early stopping
    if current_iou > best_iou:
        best_iou = current_iou
        patience_counter = 0
        
        checkpoint = {"state_dict": model.state_dict()}
        model_path = os.path.join(MODEL_SAVE_DIR, "kios_vepl_powerline_model.pth.tar")
        torch.save(checkpoint, model_path)
        
        print(f"\n✅ BEST MODEL | Loss: {train_loss:.4f} | Val IoU: {current_iou:.4f}")
    else:
        patience_counter += 1
        print(f"\nLoss: {train_loss:.4f} | Val IoU: {current_iou:.4f} | Best: {best_iou:.4f}")
        print(f"⏳ No improvement for {patience_counter} epoch(s)")
    
    # Early stopping
    if patience_counter >= early_stop_patience:
        print(f"\n🛑 Early stopping triggered after {patience_counter} epochs without improvement")
        print(f"Best IoU: {best_iou:.4f}")
        break
    
    # MPS memory management
    if DEVICE == "mps":
        torch.mps.empty_cache()
    
    print("-" * 80)

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print("=" * 70)
print(f"Best Validation IoU: {best_iou:.4f}")
print(f"Total epochs trained: {len(training_history['train_loss'])}")
print(f"Final learning rate: {optimizer.param_groups[0]['lr']:.2e}")
print(f"Model saved: kios_vepl_powerline_model.pth.tar")
print("=" * 70)


Starting training with early stopping...
Early stopping patience: 7 epochs


Epoch 1/20 | LR: 1.00e-04


Training:  69%|██████▉   | 110/160 [02:13<01:00,  1.21s/it, loss=0.7937, avg_loss=0.8221]


KeyboardInterrupt: 

## 10. Final Testing on Unseen Data

In [ ]:
print("\n" + "=" * 70)
print("FINAL TESTING ON UNSEEN DATA")
print("=" * 70)

# Create test data loader
test_loader = DataLoader(
    KIOSPowerLineDataset(test_imgs, test_msks, transform=val_transform),
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=num_workers
)

print(f"Test samples: {len(test_imgs)}")
print(f"Test batches: {len(test_loader)}")

# Load best model
print("\nLoading best model...")
checkpoint = torch.load(
    os.path.join(MODEL_SAVE_DIR, "kios_vepl_powerline_model.pth.tar"), 
    map_location=DEVICE
)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

# Test model
print("Testing model on unseen data...\n")
test_iou = check_accuracy(test_loader, model, device=DEVICE)

print(f"\n{'='*70}")
print("FINAL TEST RESULTS")
print(f"{'='*70}")
print(f"Test IoU: {test_iou:.4f}")
print(f"Validation IoU: {best_iou:.4f}")
print(f"Difference: {abs(test_iou - best_iou):.4f}")

if abs(test_iou - best_iou) < 0.05:
    print("\n✅ Good generalization (test similar to validation)")
elif test_iou < best_iou - 0.1:
    print("\n⚠️  Warning: Possible overfitting (test much worse than validation)")
else:
    print("\n✅ Model generalizes reasonably well")

print("=" * 70)

## 11. Training History Visualization

In [ ]:
def plot_training_history(history):
    """
    Plot training loss and validation IoU curves.
    """
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Training loss
    ax1.plot(epochs, history['train_loss'], 'b-', linewidth=2, marker='o')
    ax1.set_title('Training Loss', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)
    
    # Validation IoU
    ax2.plot(epochs, history['val_iou'], 'g-', linewidth=2, marker='o')
    ax2.axhline(y=max(history['val_iou']), color='r', linestyle='--', alpha=0.5, label=f'Best: {max(history["val_iou"]):.4f}')
    ax2.set_title('Validation IoU', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('IoU Score')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    # Learning rate schedule
    ax3.plot(epochs, history['learning_rates'], 'r-', linewidth=2, marker='o')
    ax3.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Learning Rate')
    ax3.set_yscale('log')
    ax3.grid(True, alpha=0.3)
    
    # Loss vs IoU scatter
    scatter = ax4.scatter(history['train_loss'], history['val_iou'], 
                         c=range(len(history['train_loss'])), 
                         cmap='viridis', s=100, alpha=0.6)
    ax4.set_title('Loss vs IoU Correlation', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Training Loss')
    ax4.set_ylabel('Validation IoU')
    ax4.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax4, label='Epoch')
    
    plt.tight_layout()
    plt.suptitle('KIOS VEPL Training Analysis', fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Print summary
    print("\nTRAINING SUMMARY")
    print("=" * 50)
    print(f"Total epochs: {len(history['train_loss'])}")
    print(f"Final training loss: {history['train_loss'][-1]:.4f}")
    print(f"Best validation IoU: {max(history['val_iou']):.4f}")
    print(f"Final validation IoU: {history['val_iou'][-1]:.4f}")
    print(f"Initial LR: {history['learning_rates'][0]:.2e}")
    print(f"Final LR: {history['learning_rates'][-1]:.2e}")
    
    # Count LR reductions
    lr_reductions = sum(1 for i in range(1, len(history['learning_rates'])) 
                       if history['learning_rates'][i] < history['learning_rates'][i-1])
    print(f"LR reductions: {lr_reductions}")

# Plot training history
plot_training_history(training_history)

## 12. Visualize Predictions

In [ ]:
def visualize_predictions(model, test_loader, num_samples=6, device="cpu"):
    """
    Visualize model predictions vs ground truth.
    """
    model.eval()
    
    # Get all data
    all_images = []
    all_masks = []
    
    with torch.no_grad():
        for images, masks in test_loader:
            for i in range(images.shape[0]):
                all_images.append(images[i])
                all_masks.append(masks[i])
    
    # Randomly select samples
    total_samples = len(all_images)
    random_indices = random.sample(range(total_samples), min(num_samples, total_samples))
    
    # Set up plot
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for sample_idx, data_idx in enumerate(random_indices):
        # Get image and mask
        image = all_images[data_idx].to(device).unsqueeze(0)
        mask = all_masks[data_idx].to(device).unsqueeze(0)
        
        # Get predictions
        prediction = model(image)
        prediction_prob = torch.sigmoid(prediction)
        predicted_mask = (prediction_prob > 0.5).float()
        
        # Convert to numpy
        image_np = image.squeeze().cpu().detach().permute(1, 2, 0).numpy()
        true_mask_np = mask.squeeze().cpu().detach().numpy()
        pred_mask_np = predicted_mask.squeeze().cpu().detach().numpy()
        prob_mask_np = prediction_prob.squeeze().cpu().detach().numpy()
        
        # Denormalize image
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image_np = std * image_np + mean
        image_np = np.clip(image_np, 0, 1)
        
        # Plot
        axes[sample_idx, 0].imshow(image_np)
        axes[sample_idx, 0].set_title(f'Image {sample_idx+1}', fontsize=12)
        axes[sample_idx, 0].axis('off')
        
        axes[sample_idx, 1].imshow(true_mask_np, cmap='gray', vmin=0, vmax=1)
        axes[sample_idx, 1].set_title('Ground Truth', fontsize=12)
        axes[sample_idx, 1].axis('off')
        
        axes[sample_idx, 2].imshow(prob_mask_np, cmap='hot', vmin=0, vmax=1)
        axes[sample_idx, 2].set_title('Probabilities', fontsize=12)
        axes[sample_idx, 2].axis('off')
        
        axes[sample_idx, 3].imshow(pred_mask_np, cmap='gray', vmin=0, vmax=1)
        axes[sample_idx, 3].set_title('Prediction', fontsize=12)
        axes[sample_idx, 3].axis('off')
    
    plt.tight_layout()
    plt.suptitle('KIOS VEPL Power Line Detection Results', fontsize=16, fontweight='bold', y=0.995)
    plt.show()
    
    print(f"Visualized {len(random_indices)} random test samples")

# Visualize predictions
print("\n" + "=" * 70)
print("PREDICTION VISUALIZATION")
print("=" * 70)

visualize_predictions(model, test_loader, num_samples=6, device=DEVICE)

print("\n✅ Training and evaluation completed!")
print("=" * 70)

## 13. Training Summary and Next Steps

In [ ]:
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(f"\n📊 Dataset:")
print(f"   Total tiles: {len(all_image_paths)}")
print(f"   Train/Val/Test: {len(train_imgs)}/{len(val_imgs)}/{len(test_imgs)}")

print(f"\n🏗️  Model:")
print(f"   Architecture: {ARCHITECTURE} + {ENCODER}")
print(f"   Parameters: {total_params:,}")

print(f"\n📈 Training:")
print(f"   Epochs completed: {len(training_history['train_loss'])}/{NUM_EPOCHS}")
print(f"   Best validation IoU: {best_iou:.4f}")
print(f"   Test IoU: {test_iou:.4f}")

print(f"\n💾 Model saved:")
print(f"   Location: {MODEL_SAVE_DIR}/kios_vepl_powerline_model.pth.tar")

print(f"\n🎯 Next Steps:")
print(f"   1. If performance is unsatisfactory:")
print(f"      - Label more images to increase dataset size")
print(f"      - Try different augmentation strategies")
print(f"      - Experiment with different architectures (U-Net, FPN)")
print(f"   2. For deployment:")
print(f"      - Test on full-size images (not just tiles)")
print(f"      - Implement inference pipeline with sliding window")
print(f"      - Optimize model for production (quantization, pruning)")
print(f"   3. For improvement:")
print(f"      - Analyze failure cases")
print(f"      - Fine-tune on difficult examples")
print(f"      - Consider ensemble methods")

print("\n" + "=" * 70)
print("✅ All tasks completed successfully!")
print("=" * 70)